In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, month, dayofweek, weekofyear, lag, avg, datediff
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer

In [0]:
df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/supply_chain_data/delta/clean")

print(f"Loaded: {df.count()} rows × {len(df.columns)} cols")

Loaded: 180519 rows × 46 cols


In [0]:
# Create date features
df = df.withColumn("order_month",month("order_date_clean"))
df = df.withColumn("order_dayofweek",dayofweek("order_date_clean"))
df = df.withColumn("order_week",weekofyear("order_date_clean"))

print("Date features created")

Date features created


In [0]:
#Shipping delay feature = Days_for_shipping_real - Days_for_shipment_scheduled
df = df.withColumn(
    "shipping_delay",
    col("Days_for_shipping_real") - col("Days_for_shipment_scheduled")
)
print("Shipping delay feature created")

Shipping delay feature created


In [0]:
#lag feature for last 7, 14, 30 days
window_spec = Window.partitionBy("Category_Name") \
                    .orderBy("order_date_clean")

df = df.withColumn("qty_lag_7",  lag("Order_Item_Quantity", 7).over(window_spec))
df = df.withColumn("qty_lag_14", lag("Order_Item_Quantity", 14).over(window_spec))
df = df.withColumn("qty_lag_30", lag("Order_Item_Quantity", 30).over(window_spec))

print("Lag features created (7, 14, 30 days)")

Lag features created (7, 14, 30 days)


In [0]:
rolling_window = Window.partitionBy("Category_Name") \
    .orderBy("order_date_clean") \
    .rowsBetween(-28, -1)

df = df.withColumn("qty_rolling_28d_avg",
                   avg("Order_Item_Quantity").over(rolling_window))

print("Rolling 28-day average created")

Rolling 28-day average created


In [0]:
# Encode categorical columns to numerical values
cat_cols    = ["Delivery_Status", "Market", "Department_Name",
               "Category_Name", "Shipping_Mode", "Type"]
output_cols = [c + "_idx" for c in cat_cols]

indexer = StringIndexer(
    inputCols=cat_cols,
    outputCols=output_cols,
    handleInvalid="keep"
)
df = indexer.fit(df).transform(df)
print(f"Encoded {len(cat_cols)} categorical columns")

Encoded 6 categorical columns


In [0]:
before = df.count()
df = df.dropna(subset=["qty_lag_7", "qty_lag_14", 
                        "qty_lag_30", "qty_rolling_28d_avg"])
after = df.count()
print(f"Dropped {before - after} rows with null lag values")

Dropped 0 rows with null lag values


In [0]:
print(f"\n Feature table: {df.count()} rows × {len(df.columns)} cols")

feature_cols = [
    "Product_Price", "order_month", "order_dayofweek", "order_week",
    "shipping_delay", "Late_delivery_risk",
    "qty_lag_7", "qty_lag_14", "qty_lag_30", "qty_rolling_28d_avg",
    "Delivery_Status_idx", "Market_idx", "Department_Name_idx",
    "Category_Name_idx", "Shipping_Mode_idx", "Type_idx"
]
print(f"\n Final ML features ({len(feature_cols)}):")
for f in feature_cols:
    print(f"{f}")

# Save feature Delta
df.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/supply_chain_data/delta/features")
print("\n Feature Delta table saved")


 Feature table: 179019 rows × 60 cols

 Final ML features (16):
Product_Price
order_month
order_dayofweek
order_week
shipping_delay
Late_delivery_risk
qty_lag_7
qty_lag_14
qty_lag_30
qty_rolling_28d_avg
Delivery_Status_idx
Market_idx
Department_Name_idx
Category_Name_idx
Shipping_Mode_idx
Type_idx

 Feature Delta table saved
